In [ ]:
!pip install optuna

In [ ]:
#@title Libs
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np'
from sklearn import metrics
from sklearn.metrics import classification_report, confusion_matrix

import seaborn
import plotly
from tqdm import tqdm

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState
from optuna.pruners import MedianPruner
import torch.optim as optim



In [ ]:
#@title Dataset Setup
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    train_dataset, [train_size, val_size]
)

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, num_classes, activation_function):
        super(MLP, self).__init__()
        self.activation_function = activation_function
        self.fc_input = nn.Linear(input_size, 64)
        self.fc_hidden1 = nn.Linear(64, 128)
        self.fc_hidden2 = nn.Linear(128, 64)
        self.fc_output = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.activation_function(self.fc_input(x))
        x = self.activation_function(self.fc_hidden1(x))
        x = self.activation_function(self.fc_hidden2(x))
        x = self.fc_output(x)
        return x

In [ ]:
#@title Modelo MLP com Dropout
class MLP_Dropout(nn.Module):
    def __init__(self, input_size, num_classes, dropout_rate=0.3):
        super(MLP_Dropout, self).__init__()
        self.fc_input = nn.Linear(input_size, 64)
        self.fc_hidden1 = nn.Linear(64, 128)
        self.fc_hidden2 = nn.Linear(128, 64)
        self.fc_output = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(dropout_rate)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc_input(x))
        x = self.dropout(x)
        x = self.relu(self.fc_hidden1(x))
        x = self.dropout(x)
        x = self.relu(self.fc_hidden2(x))
        x = self.dropout(x)
        x = self.fc_output(x)
        return x

In [ ]:
class DeepMLP_Dropout(nn.Module):
    def __init__(self, input_size, num_classes, dropout_rate=0.5):
        super(DeepMLP_Dropout, self).__init__()
        # Arquitetura: 3072 → 512 → 256 → 128 → 64 → 10
        self.fc1 = nn.Linear(input_size, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc_output = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(dropout_rate)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.relu(self.fc4(x))
        x = self.dropout(x)
        x = self.fc_output(x)
        return x

In [ ]:
#@title Modelo MLP Dinâmico (para otimização completa)
class DynamicMLP(nn.Module):
    def __init__(self, input_size, num_classes, hidden_sizes, dropout_rate=0.3):
        super(DynamicMLP, self).__init__()
        layers = []
        prev_size = input_size

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
#@title Função de Treinamento
def train_model(model, train_loader, val_loader, optimizer, loss_function,
                num_epochs, device, patience=10, trial=None):
    best_val_acc = 0.0
    patience_counter = 0

    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        for images, labels in train_loader:
            images = images.view(-1, 32*32*3).to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.view(-1, 32*32*3).to(device)
                labels = labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = val_correct / val_total

        # Report to Optuna
        if trial is not None:
            trial.report(val_acc, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    return best_val_acc

In [ ]:
#@title Função de Avaliação
def evaluate_model(model, test_loader, device):
    model.eval()
    predictions = []
    labels_list = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.view(-1, 32*32*3).to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            predictions.extend(predicted.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())

    return {
        "accuracy": metrics.accuracy_score(labels_list, predictions),
        "balanced_accuracy": metrics.balanced_accuracy_score(labels_list, predictions),
        "precision": metrics.precision_score(labels_list, predictions, average="weighted"),
        "recall": metrics.recall_score(labels_list, predictions, average="weighted"),
        "f1_score": metrics.f1_score(labels_list, predictions, average="weighted")
    }

In [ ]:
#@title EXPERIMENTO 1: Variação do Batch Size
print("="*60)
print("EXPERIMENTO 1: Variação do Batch Size com Optuna")
print("="*60)

EXPERIMENTO 1: Variação do Batch Size com Optuna


In [ ]:
def objective_batch_size(trial):
    # Hiperparâmetros para otimizar
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)

    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Data loaders
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Model
    model = MLP_Dropout(input_size=32*32*3, num_classes=10, dropout_rate=dropout_rate).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()

    # Train
    val_acc = train_model(model, train_loader, val_loader, optimizer, loss_function,
                          num_epochs=30, device=device, patience=3, trial=trial)

    return val_acc

In [ ]:
# Criar estudo Optuna
study1 = optuna.create_study(direction='maximize', study_name='batch_size_optimization')
study1.optimize(objective_batch_size, n_trials=5, show_progress_bar=True)

print("\nMelhores Hiperparâmetros (Experimento 1):")
print(study1.best_params)
print(f"Melhor Acurácia de Validação: {study1.best_value:.4f}")


[I 2025-09-30 01:10:21,534] A new study created in memory with name: batch_size_optimization


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2025-09-30 01:13:45,289] Trial 0 finished with value: 0.516 and parameters: {'batch_size': 256, 'learning_rate': 0.0029369067674771907, 'dropout_rate': 0.1198971812909008}. Best is trial 0 with value: 0.516.
[I 2025-09-30 01:16:03,098] Trial 1 finished with value: 0.5098 and parameters: {'batch_size': 32, 'learning_rate': 0.0011303289390797177, 'dropout_rate': 0.13311007321104143}. Best is trial 0 with value: 0.516.
[I 2025-09-30 01:17:30,422] Trial 2 finished with value: 0.4584 and parameters: {'batch_size': 256, 'learning_rate': 0.0067430447287420685, 'dropout_rate': 0.148661614594078}. Best is trial 0 with value: 0.516.
[I 2025-09-30 01:19:06,501] Trial 3 finished with value: 0.368 and parameters: {'batch_size': 256, 'learning_rate': 0.007752338324270162, 'dropout_rate': 0.3749530137894165}. Best is trial 0 with value: 0.516.
[I 2025-09-30 01:21:08,382] Trial 4 finished with value: 0.3952 and parameters: {'batch_size': 32, 'learning_rate': 0.0022996592320369716, 'dropout_rate': 0

In [ ]:
#@title EXPERIMENTO 2: Rede Mais Profunda com Dropout Forte
print("\n" + "="*60)
print("EXPERIMENTO 2: Rede Mais Profunda com Dropout Forte")
print("="*60)


EXPERIMENTO 2: Rede Mais Profunda com Dropout Forte


In [ ]:
def objective_deep_network(trial):
    # Hiperparâmetros para otimizar
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.4, 0.7)  # Dropout forte
    batch_size = 64  # Fixo para este experimento

    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Data loaders
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Model - Arquitetura: 3072 → 512 → 256 → 128 → 64 → 10
    model = DeepMLP_Dropout(input_size=32*32*3, num_classes=10, dropout_rate=dropout_rate).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()

    # Train
    val_acc = train_model(model, train_loader, val_loader, optimizer, loss_function,
                          num_epochs=30, device=device, patience=10, trial=trial)

    return val_acc


In [ ]:
study2 = optuna.create_study(direction='maximize', study_name='deep_network_optimization')
study2.optimize(objective_deep_network, n_trials=5, show_progress_bar=True)

print("\nMelhores Hiperparâmetros (Experimento 2):")
print(study2.best_params)
print(f"Melhor Acurácia de Validação: {study2.best_value:.4f}")


[I 2025-09-30 01:21:08,403] A new study created in memory with name: deep_network_optimization


  0%|          | 0/5 [00:00<?, ?it/s]

[I 2025-09-30 01:27:18,552] Trial 0 finished with value: 0.5402 and parameters: {'learning_rate': 0.000264527575710669, 'dropout_rate': 0.48093747334136194}. Best is trial 0 with value: 0.5402.
[I 2025-09-30 01:33:40,404] Trial 1 finished with value: 0.5184 and parameters: {'learning_rate': 0.0001119365558559686, 'dropout_rate': 0.5313813797466895}. Best is trial 0 with value: 0.5402.
[I 2025-09-30 01:39:54,053] Trial 2 finished with value: 0.456 and parameters: {'learning_rate': 0.00027969332670689664, 'dropout_rate': 0.6552351705618015}. Best is trial 0 with value: 0.5402.
[I 2025-09-30 01:46:04,697] Trial 3 finished with value: 0.4668 and parameters: {'learning_rate': 0.0005283837403415454, 'dropout_rate': 0.6281279663412195}. Best is trial 0 with value: 0.5402.
[I 2025-09-30 01:52:16,441] Trial 4 finished with value: 0.525 and parameters: {'learning_rate': 0.00012986960033155821, 'dropout_rate': 0.5169142972521171}. Best is trial 0 with value: 0.5402.

Melhores Hiperparâmetros (Exp

In [ ]:
#@title EXPERIMENTO 3: Otimização Completa de Todos os Parâmetros
print("\n" + "="*60)
print("EXPERIMENTO 3: Otimização Completa com Optuna")
print("="*60)


EXPERIMENTO 3: Otimização Completa com Optuna


In [ ]:
def objective_complete(trial):
    # Hiperparâmetros para otimizar
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.6)

    # Arquitetura dinâmica
    n_layers = trial.suggest_int('n_layers', 2, 4)
    hidden_sizes = []
    for i in range(n_layers):
        size = trial.suggest_categorical(f'hidden_size_{i}', [64, 128, 256, 512])
        hidden_sizes.append(size)

    # Device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Data loaders
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Model
    model = DynamicMLP(input_size=32*32*3, num_classes=10,
                       hidden_sizes=hidden_sizes, dropout_rate=dropout_rate).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()

    # Train
    val_acc = train_model(model, train_loader, val_loader, optimizer, loss_function,
                          num_epochs=30, device=device, patience=10, trial=trial)

    return val_acc

In [ ]:
# Criar estudo Optuna
study3 = optuna.create_study(direction='maximize', study_name='complete_optimization')
study3.optimize(objective_complete, n_trials=8, show_progress_bar=True)

[I 2025-09-30 01:52:16,463] A new study created in memory with name: complete_optimization


  0%|          | 0/8 [00:00<?, ?it/s]

[I 2025-09-30 01:58:19,544] Trial 0 finished with value: 0.572 and parameters: {'batch_size': 64, 'learning_rate': 0.0003581922683707069, 'dropout_rate': 0.29006926262275623, 'n_layers': 2, 'hidden_size_0': 512, 'hidden_size_1': 256}. Best is trial 0 with value: 0.572.
[I 2025-09-30 02:03:06,507] Trial 1 finished with value: 0.3276 and parameters: {'batch_size': 64, 'learning_rate': 0.0062025950639402255, 'dropout_rate': 0.25596101433595175, 'n_layers': 3, 'hidden_size_0': 64, 'hidden_size_1': 512, 'hidden_size_2': 256}. Best is trial 0 with value: 0.572.
[I 2025-09-30 02:09:57,987] Trial 2 finished with value: 0.5162 and parameters: {'batch_size': 32, 'learning_rate': 0.0002928247823128969, 'dropout_rate': 0.5270885244306611, 'n_layers': 2, 'hidden_size_0': 128, 'hidden_size_1': 128}. Best is trial 0 with value: 0.572.
[I 2025-09-30 02:15:22,773] Trial 3 finished with value: 0.2424 and parameters: {'batch_size': 32, 'learning_rate': 0.00395956880298469, 'dropout_rate': 0.4851382564264

In [ ]:
print("\nMelhores Hiperparâmetros (Experimento 3):")
print(study3.best_params)
print(f"Melhor Acurácia de Validação: {study3.best_value:.4f}")


Melhores Hiperparâmetros (Experimento 3):
{'batch_size': 64, 'learning_rate': 0.0003581922683707069, 'dropout_rate': 0.29006926262275623, 'n_layers': 2, 'hidden_size_0': 512, 'hidden_size_1': 256}
Melhor Acurácia de Validação: 0.5720


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# Experimento 1 - Melhor modelo
print("\nExperimento 1 - Variação do Batch Size:")
best_params_1 = study1.best_params
model_1 = MLP_Dropout(32*32*3, 10, best_params_1['dropout_rate']).to(device)
train_loader_1 = torch.utils.data.DataLoader(
    torch.utils.data.ConcatDataset([train_dataset, val_dataset]),
    batch_size=best_params_1['batch_size'], shuffle=True
)
optimizer_1 = torch.optim.Adam(model_1.parameters(), lr=best_params_1['learning_rate'])
train_model(model_1, train_loader_1, test_loader, optimizer_1, nn.CrossEntropyLoss(),
            num_epochs=30, device=device, patience=10)
scores_1 = evaluate_model(model_1, test_loader, device)
print("Resultados:", scores_1)

# Experimento 2 - Melhor modelo
print("\nExperimento 2 - Rede Profunda com Dropout:")
best_params_2 = study2.best_params
model_2 = DeepMLP_Dropout(32*32*3, 10, best_params_2['dropout_rate']).to(device)
train_loader_2 = torch.utils.data.DataLoader(
    torch.utils.data.ConcatDataset([train_dataset, val_dataset]),
    batch_size=64, shuffle=True
)
optimizer_2 = torch.optim.Adam(model_2.parameters(), lr=best_params_2['learning_rate'])
train_model(model_2, train_loader_2, test_loader, optimizer_2, nn.CrossEntropyLoss(),
            num_epochs=30, device=device, patience=10)
scores_2 = evaluate_model(model_2, test_loader, device)
print("Resultados:", scores_2)

# Experimento 3 - Melhor modelo
print("\nExperimento 3 - Otimização Completa:")
best_params_3 = study3.best_params
hidden_sizes_3 = [best_params_3[f'hidden_size_{i}']
                  for i in range(best_params_3['n_layers'])]
model_3 = DynamicMLP(32*32*3, 10, hidden_sizes_3, best_params_3['dropout_rate']).to(device)
train_loader_3 = torch.utils.data.DataLoader(
    torch.utils.data.ConcatDataset([train_dataset, val_dataset]),
    batch_size=best_params_3['batch_size'], shuffle=True
)
optimizer_3 = torch.optim.Adam(model_3.parameters(), lr=best_params_3['learning_rate'])
train_model(model_3, train_loader_3, test_loader, optimizer_3, nn.CrossEntropyLoss(),
            num_epochs=20, device=device, patience=7)
scores_3 = evaluate_model(model_3, test_loader, device)
print("Resultados:", scores_3)

print("\n" + "="*60)
print("COMPARAÇÃO FINAL DOS 3 EXPERIMENTOS")
print("="*60)
print(f"Experimento 1 - Accuracy: {scores_1['accuracy']:.4f}")
print(f"Experimento 2 - Accuracy: {scores_2['accuracy']:.4f}")
print(f"Experimento 3 - Accuracy: {scores_3['accuracy']:.4f}")


Experimento 1 - Variação do Batch Size:
Resultados: {'accuracy': 0.5111, 'balanced_accuracy': np.float64(0.5111), 'precision': 0.5176000778537506, 'recall': 0.5111, 'f1_score': 0.5093699405042259}

Experimento 2 - Rede Profunda com Dropout:
Resultados: {'accuracy': 0.546, 'balanced_accuracy': np.float64(0.5459999999999999), 'precision': 0.5463588420214188, 'recall': 0.546, 'f1_score': 0.5451374291107187}

Experimento 3 - Otimização Completa:
Resultados: {'accuracy': 0.5733, 'balanced_accuracy': np.float64(0.5732999999999999), 'precision': 0.5731013852365214, 'recall': 0.5733, 'f1_score': 0.5715814334040166}

COMPARAÇÃO FINAL DOS 3 EXPERIMENTOS
Experimento 1 - Accuracy: 0.5111
Experimento 2 - Accuracy: 0.5460
Experimento 3 - Accuracy: 0.5733


In [ ]:
#@title Análise Detalhada - Experimento 3
import seaborn as sns

# Classes do CIFAR-10
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Re-avaliar modelo 3 para pegar labels e predictions
print("="*70)
print("RELATÓRIO DE CLASSIFICAÇÃO - EXPERIMENTO 3 (MELHOR MODELO)")
print("="*70)

model_3.eval()
all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(-1, 32*32*3).to(device)
        labels = labels.to(device)
        outputs = model_3(images)
        _, predicted = torch.max(outputs, 1)
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Relatório por classe
print(classification_report(all_labels, all_predictions,
                           target_names=class_names, digits=4))

# Matriz de Confusão
cm = confusion_matrix(all_labels, all_predictions)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Número de Predições'})
plt.title('Matriz de Confusão - Experimento 3', fontsize=14, fontweight='bold')
plt.ylabel('Classe Verdadeira', fontsize=12)
plt.xlabel('Classe Predita', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# -------------------------------------------------------------
#  Experimento 4 — Adam vs SGD + LR + BatchSize + Dropout
# -------------------------------------------------------------

In [ ]:
def objective_adam_sgd(trial):
    # Espaço de busca
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    dropout_rate = trial.suggest_float("dropout", 0.0, 0.5, step=0.1)
    momentum = 0.0
    if optimizer_name == "SGD":
        momentum = trial.suggest_categorical("momentum", [0.0, 0.9])

    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Modelo
    model = MLP(input_size=input_size, num_classes=num_classes,
                activation_function=activation_function, dropout_rate=dropout_rate)

    # Otimizador
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)

    # Treino
    accuracy = train_model(model, optimizer, train_loader, test_loader,
                           loss_function, epochs=num_epochs)
    return accuracy




# -------------------------------------------------------------
# 🔹 Rodar estudo Optuna
# -------------------------------------------------------------

In [ ]:
from tqdm.notebook import tqdm




n_trials = 20
study_adam_sgd = optuna.create_study(direction="maximize")

# Barra externa: progresso geral das trials
with tqdm(total=n_trials, desc="Optuna Trials") as pbar_trials:
    for _ in range(n_trials):
        # Cada trial roda apenas 1 vez
        study_adam_sgd.optimize(objective_adam_sgd, n_trials=1)

        # Atualiza a barra externa após cada trial
        pbar_trials.update(1)

def print_callback(study, trial):
    print(f"Trial {trial.number}: Accuracy = {trial.value:.4f} | Params = {trial.params}")

print("Melhor configuração:", study_adam_sgd.best_trial.params)
print("Melhor acurácia:", study_adam_sgd.best_value)

# Visualização
ov.plot_optimization_history(study_adam_sgd)
ov.plot_contour(study_adam_sgd)

In [ ]:
#@title Experimento 5 -  Weight_decay

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
#@title Dataset Setup

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)


In [ ]:
#@title Defining the MLP model
# 3072 (input) → 64 → 128 → 64 → 10 (output)

class MLP(nn.Module):
  def __init__(self, input_size, num_classes, activation_function, dropout_rate: float = 0.0):
    super(MLP,self).__init__()
    # Defining activation functions and fully-connected layers
    self.activation_function = activation_function
    self.fc_input = nn.Linear(input_size, 64)
    self.fc_hidden1 = nn.Linear(64, 128)
    self.fc_hidden2 = nn.Linear(128, 64)
    self.fc_output = nn.Linear(64, num_classes)
    self.drop = nn.Dropout(p=dropout_rate)

  def forward(self,x):
    x = self.activation_function(self.fc_input(x));   x = self.drop(x)
    x = self.activation_function(self.fc_hidden1(x)); x = self.drop(x)
    x = self.activation_function(self.fc_hidden2(x)); x = self.drop(x)
    x = self.fc_output(x)
    return x

In [ ]:
#@title Defining metrics helper

def get_scores(targets, predictions):
    return {
        "accuracy": metrics.accuracy_score(targets, predictions),
        "balanced_accuracy": metrics.balanced_accuracy_score(targets, predictions),
        "precision": metrics.precision_score(targets, predictions, average="weighted"),
        "recall": metrics.recall_score(targets, predictions, average="weighted"),
        "f1_score": metrics.f1_score(targets, predictions, average="weighted")
    }

In [ ]:
#@title Hyperparameters
input_size = 32*32*3 # 32x32 RGB images
num_classes = 10

learning_rate = 0.001
num_epochs = 100
batch_size = 16
activation_function = nn.ReLU()

loss_function = nn.CrossEntropyLoss()

weight_decay = 1e-4
dropout_rate = 0.3


In [ ]:
#@title Loaders

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:

def objective(trial):
    lr         = trial.suggest_float("lr", 5e-4, 3e-3, log=True)
    bs         = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    dr         = trial.suggest_float("dropout", 0.0, 0.6)
    wd         = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = torch.utils.data.DataLoader(test_dataset,  batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)

    model = MLP(input_size=input_size, num_classes=num_classes,
                activation_function=activation_function, dropout_rate=dr).to(device)
    optimz = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    crit   = loss_function

    max_epochs = 10
    for epoch in range(1, max_epochs+1):
        model.train()
        for images, labels in train_loader:
            images = images.view(-1, 32*32*3).to(device)
            labels = labels.to(device)
            optimz.zero_grad()
            out = model(images)
            loss = crit(out, labels)
            loss.backward()
            optimz.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.view(-1, 32*32*3).to(device)
                labels = labels.to(device)
                pred = model(images).argmax(1)
                correct += (pred == labels).sum().item()
                total   += labels.size(0)
        val_acc = correct / total

        trial.report(val_acc, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return val_acc

sampler = TPESampler(seed=42, multivariate=True, group=True)
pruner  = MedianPruner(n_warmup_steps=5)
study   = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)


study.optimize(objective, n_trials=3, show_progress_bar=True)
print("Best val_acc:", study.best_value)
print("Best params:", study.best_params)

best = study.best_params
learning_rate = best["lr"]
batch_size    = best["batch_size"]
dropout_rate  = best["dropout"]
weight_decay  = best["weight_decay"]

In [ ]:
# === treino final (100 épocas) ===
best = study.best_params
learning_rate = best["lr"]
batch_size    = best["batch_size"]
dropout_rate  = best["dropout"]
weight_decay  = best["weight_decay"]

num_epochs = 100  #

use_cuda = torch.cuda.is_available()
PIN = bool(use_cuda)
WORKERS = 0 if not use_cuda else 2

from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=WORKERS, pin_memory=PIN)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=WORKERS, pin_memory=PIN)


In [ ]:
#@title Training loop

# Build the model
mlp = MLP(input_size=input_size, num_classes=num_classes,
          activation_function=activation_function,
          dropout_rate=dropout_rate).to(device)

# Setting optimizer up
optimizer = torch.optim.Adam(mlp.parameters(), lr=learning_rate,
                             weight_decay=weight_decay)  # <<< usa weight_decay do Optuna

# Early stopping setup
best_loss = float('inf')
patience = 5
patience_counter = 0

# Start training epochs loop
for epoch in tqdm(range(num_epochs)):
  epoch_loss = 0.0
  for i, (images, labels) in enumerate(train_loader):
    images = images.view(-1,32*32*3).to(device) # flattenning images
    labels = labels.to(device)

    # Forward pass
    optimizer.zero_grad()
    outputs = mlp(images)

    # Backward pass
    loss = loss_function(outputs, labels)
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()

    if (i+1) % 1000 == 0:
      tqdm.write(f' Epoch {epoch + 1}/{num_epochs}, Step {i+1}/{len(train_dataset) // batch_size}, Loss: {loss}')

  epoch_loss /= len(train_loader)
  tqdm.write(f'Epoch {epoch+1} average loss: {epoch_loss:.4f}')

  # Early stopping using loss value
  if epoch_loss < best_loss:
    best_loss = epoch_loss
    patience_counter = 0
  else:
    patience_counter += 1
    if patience_counter >= patience:
      tqdm.write("Early stopping triggered.")
      break

In [ ]:
#@title Evaluate model (accuracy, precision, recall)
mlp.eval()
predictions = []
labels = []
for images, label in test_loader:
  images = images.view(-1,32*32*3).to(device)
  label = label.to(device)

  output = mlp(images)
  _, predicted = torch.max(output,1)

  predictions.extend(predicted.cpu().numpy())
  labels.extend(label.cpu().numpy())

scores = get_scores(labels, predictions)
print("Scores of your model\n", scores)


try:
    class_names = test_dataset.classes  # ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
except Exception:
    class_names = None

# relatório por classe (precision, recall, f1) e métricas globais
print("\nClassification report:\n")
print(classification_report(
    labels,                        # verdadeiros
    predictions,                   # previstos
    target_names=class_names if class_names else None,
    digits=3
))

# matriz de confusão
cm = confusion_matrix(labels, predictions)
print("\nConfusion matrix:\n", cm)

In [ ]:
#@title Experimento 6 -Função de Ativação ReLU vs Tanh

In [ ]:
#@title Dataset Setup

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)


In [ ]:
#@title Defining the MLP model
# 3072 (input) → 64 → 128 → 64 → 10 (output)

class MLP(nn.Module):
  def __init__(self, input_size, num_classes, activation_function):
    super(MLP, self).__init__()
    self.activation_function = activation_function
    # Defining activationfunctions and fully-connected layers
    self.fc_input = nn.Linear(input_size, 512)
    self.fc_hidden1 = nn.Linear(512, 256)
    self.fc_hidden2 = nn.Linear(256, 128)
    self.fc_hidden3 = nn.Linear(128, 64)
    self.fc_hidden4 = nn.Linear(64, 32)
    self.fc_output = nn.Linear(32, num_classes)

  def forward(self, x):
    x = self.activation_function(self.fc_input(x))
    x = self.activation_function(self.fc_hidden1(x))
    x = self.activation_function(self.fc_hidden2(x))
    x = self.activation_function(self.fc_hidden3(x))
    x = self.activation_function(self.fc_hidden4(x))
    x = self.fc_output(x)
    return x

In [ ]:
#@title Defining metrics helper

def get_scores(targets, predictions):
    return {
        "accuracy": metrics.accuracy_score(targets, predictions),
        "balanced_accuracy": metrics.balanced_accuracy_score(targets, predictions),
        "precision": metrics.precision_score(targets, predictions, average="weighted"),
        "recall": metrics.recall_score(targets, predictions, average="weighted"),
        "f1_score": metrics.f1_score(targets, predictions, average="weighted")
    }

In [ ]:
#@title Hyperparameters
input_size = 32*32*3  # 32x32 RGB images
num_classes = 10

loss_function = nn.CrossEntropyLoss()

In [ ]:
#@title Loaders

batch_size = 128

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:

def objective(trial):
    # Hiperparâmetros a serem otimizados
    activation_name = trial.suggest_categorical("activation", ["ReLU", "Tanh"])
    if activation_name == "ReLU":
        activation_function = nn.ReLU()
    else:
        activation_function = nn.Tanh()

    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-1)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])

    # Criar dataloaders (train/val split)
    val_size = int(0.2 * len(train_dataset))
    train_size = len(train_dataset) - val_size
    train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    # Modelo, loss e otimizador
    model = MLP(input_size, num_classes, activation_function).to("cuda" if torch.cuda.is_available() else "cpu")
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Treino simples (poucas épocas para Optuna ser rápido)
    num_epochs = 10
    for epoch in range(num_epochs):
        model.train()
        for images, labels in train_loader:
            images = images.view(-1, 32*32*3).to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

    # Validação
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.view(-1, 32*32*3).to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    scores = get_scores(all_labels, all_preds)
    return scores["accuracy"]

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)  # pode aumentar n_trials

print("Melhor trial:")
trial = study.best_trial
print(trial.params)


In [ ]:
# Melhor trial encontrado pelo Optuna
best_params = study.best_trial.params
print("Melhores parâmetros do Optuna:", best_params)

# Definir função de ativação
if best_params["activation"] == "ReLU":
    activation_function = nn.ReLU()
else:
    activation_function = nn.Tanh()

# Criar DataLoader com batch_size do Optuna
test_loader = DataLoader(test_dataset, batch_size=best_params["batch_size"], shuffle=False)

# Criar modelo com os melhores parâmetros
device = "cuda" if torch.cuda.is_available() else "cpu"
mlp = MLP(input_size, num_classes, activation_function).to(device)
optimizer = torch.optim.Adam(mlp.parameters(), lr=best_params["learning_rate"])
loss_function = nn.CrossEntropyLoss()

# Treinar novamente o modelo final (usar mais épocas, agora que sabemos os melhores hiperparâmetros)
num_epochs = 20
for epoch in range(num_epochs):
    mlp.train()
    for images, labels in DataLoader(train_dataset, batch_size=best_params["batch_size"], shuffle=True):
        images = images.view(-1, 32*32*3).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = mlp(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

# Avaliação no conjunto de teste
mlp.eval()
predictions, labels = [], []
with torch.no_grad():
    for images, label in test_loader:
        images = images.view(-1, 32*32*3).to(device)
        label = label.to(device)

        output = mlp(images)
        _, predicted = torch.max(output, 1)

        predictions.extend(predicted.cpu().numpy())
        labels.extend(label.cpu().numpy())

scores = get_scores(labels, predictions)
print("Scores no conjunto de teste:\n", scores)


In [ ]:
#@title Evaluate model (accuracy, precision, recall)
mlp.eval()
predictions = []
labels = []
for images, label in test_loader:
  images = images.view(-1,32*32*3).cuda()
  label = label.cuda()

  output = mlp(images)
  _, predicted = torch.max(output,1)

  predictions.extend(predicted.cpu().numpy())
  labels.extend(label.cpu().numpy())

scores = get_scores(labels, predictions)
print("Scores of your model\n", scores)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(labels, predictions)

# Display the confusion matrix using a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=train_dataset.classes, yticklabels=train_dataset.classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

#@ Experimento 7 - Função de Ativação ReLU vs Sigmoid (Optuna)

In [ ]:
#@title Dataset Setup

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)


In [ ]:
from torch.utils.data import random_split

train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

In [ ]:
#@title Defining the MLP model
# 3072 (input) → 64 → 128 → 64 → 10 (output)

class MLP(nn.Module):
  def __init__(self, input_size, num_classes, activation_function):
    super(MLP, self).__init__()
    self.activation_function = activation_function
    # Defining activationfunctions and fully-connected layers
    self.fc_input = nn.Linear(input_size, 512)
    self.fc_hidden1 = nn.Linear(512, 256)
    self.fc_hidden2 = nn.Linear(256, 128)
    self.fc_hidden3 = nn.Linear(128, 64)
    self.fc_hidden4 = nn.Linear(64, 32)
    self.fc_output = nn.Linear(32, num_classes)

  def forward(self, x):
    x = self.activation_function(self.fc_input(x))
    x = self.activation_function(self.fc_hidden1(x))
    x = self.activation_function(self.fc_hidden2(x))
    x = self.activation_function(self.fc_hidden3(x))
    x = self.activation_function(self.fc_hidden4(x))
    x = self.fc_output(x)
    return x

In [ ]:
#@title Defining metrics helper

def get_scores(targets, predictions):
    return {
        "accuracy": metrics.accuracy_score(targets, predictions),
        "balanced_accuracy": metrics.balanced_accuracy_score(targets, predictions),
        "precision": metrics.precision_score(targets, predictions, average="weighted"),
        "recall": metrics.recall_score(targets, predictions, average="weighted"),
        "f1_score": metrics.f1_score(targets, predictions, average="weighted")
    }

In [ ]:
#@title Hyperparameters
input_size = 32*32*3  # 32x32 RGB images
num_classes = 10

loss_function = nn.CrossEntropyLoss()

In [ ]:
#@title Loaders

batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def train(model, loader, criterion, optimizer, device):
    # treina a reed neural, de forma a iterar sobre os dados realizando um foward,
    # calcula a perda e atualiza os pesos do modelo usando backpropagation e otimizadores
    model.train()

    for images, labels in loader:
        images = images.view(-1, 32*32*3).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

def evaluate(model, loader, device):
    # avalia o desempenho do modelo treinado, iterando sobre os dados fazendo previsões e retornando a acurácia do modelo
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.view(-1, 32*32*3).to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

In [ ]:

def objective(trial):
  # --- escolher hiperparâmetros ---
  activation_name = trial.suggest_categorical("activation", ["ReLU", "Sigmoid"])
  if activation_name == "ReLU":
    activation_function = nn.ReLU()
  else:
    activation_function = nn.Sigmoid()

  learning_rate = trial.suggest_loguniform("lr", 1e-4, 1e-2)
  batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

  # --- definir modelo ---
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model = MLP(input_size=3072, num_classes=10, activation_function=activation_function).to(device)

  optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
  criterion = nn.CrossEntropyLoss()

  # --- criar dataloaders com batch_size ---
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

  # --- treinar ---
  for epoch in range(5):  # pode aumentar se quiser
    train(model, train_loader, criterion, optimizer, device)

  # --- avaliar ---
  acc = evaluate(model, val_loader, device)

  return acc

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best trial:")
trial = study.best_trial
print(f"Acurácia: {trial.value}")
print("Hiperparâmetros:", trial.params)

In [ ]:
# Melhor trial encontrado pelo Optuna
best_params = study.best_trial.params
print("Melhores parâmetros do Optuna:", best_params)

# Definir função de ativação
if best_params["activation"] == "ReLU":
    activation_function = nn.ReLU()
else:
    activation_function = nn.Tanh()

# Criar DataLoader com batch_size do Optuna
test_loader = DataLoader(test_dataset, batch_size=best_params["batch_size"], shuffle=False)

# Criar modelo com os melhores parâmetros
device = "cuda" if torch.cuda.is_available() else "cpu"
mlp = MLP(input_size, num_classes, activation_function).to(device)
optimizer = torch.optim.Adam(mlp.parameters(), lr=best_params["lr"])
loss_function = nn.CrossEntropyLoss()

# Treinar novamente o modelo final (usar mais épocas, agora que sabemos os melhores hiperparâmetros)
num_epochs = 20
for epoch in range(num_epochs):
    mlp.train()
    for images, labels in DataLoader(train_dataset, batch_size=best_params["batch_size"], shuffle=True):
        images = images.view(-1, 32*32*3).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = mlp(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

# Avaliação no conjunto de teste
mlp.eval()
predictions, labels = [], []
with torch.no_grad():
    for images, label in test_loader:
        images = images.view(-1, 32*32*3).to(device)
        label = label.to(device)

        output = mlp(images)
        _, predicted = torch.max(output, 1)

        predictions.extend(predicted.cpu().numpy())
        labels.extend(label.cpu().numpy())

scores = get_scores(labels, predictions)
print("Scores no conjunto de teste:\n", scores)


In [ ]:
#@title Evaluate model (accuracy, precision, recall)
mlp.eval()
predictions = []
labels = []
for images, label in test_loader:
  images = images.view(-1,32*32*3).cuda()
  label = label.cuda()

  output = mlp(images)
  _, predicted = torch.max(output,1)

  predictions.extend(predicted.cpu().numpy())
  labels.extend(label.cpu().numpy())

scores = get_scores(labels, predictions)
print("Scores of your model\n", scores)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(labels, predictions)

# Display the confusion matrix using a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=test_dataset.classes, yticklabels=test_dataset.classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()